# Trying to break the laws

Each cell attempts a move the calculus forbids and shows the law that stops it.

Prerequisites: rational arithmetic and directed graphs.
Objectives: compute both diagnostics; distinguish abstract law checks from measured replay; identify the witness or hypothesis that a rejected step lacks.


In [1]:
import json
import sys
from pathlib import Path

# Locate the package from wherever the kernel starts, then import the flat modules.
here = Path.cwd().resolve()
candidates = (here, *here.parents, here / "omega_engine_atlas/coda_atlas/omega_engine_discrete_deformation_calculus")
root = next(p for p in candidates if (p / "discrete_deformation_calculus.md").is_file())
sys.path.insert(0, str(root / "code" / "ddc"))
metadata = json.loads((root / "zenodo_metadata.json").read_text(encoding="utf-8"))
print("Discrete Deformation Calculus", metadata["version"])

Discrete Deformation Calculus 1.0.3


## 1. Every falsifier among the demos is refused

In [2]:
import demos

for demo in demos.all_demos():
    if not demo.lawful:
        print(f"{demo.name:34s} halted at step {demo.run.halted_at}: {', '.join(demo.run.violations)}")
        assert demo.run.violations

demo1_open_CRET_unlawful           halted at step 2: ExactRepairFromNonExactClass
demo2_CRMT_unlawful                halted at step 3: TerminalizedOpenResidual
demo3_face_lift_then_T_unlawful    halted at step 3: TerminalizedOpenResidual
demo3_unlicensed_lift_unlawful     halted at step 2: UnlicensedLift
demo4_CMT_unlawful                 halted at step 2: BlindDetector
demo5_DIT_unlawful                 halted at step 0: MissingLossLedger


## 2. A class-crossing edit passes the deposited historical comparison law and fails the paper law

In [3]:
from fractions import Fraction as F
import automaton as A

open_state = A.State("p", (F(1),), 1, 1)
edited = A.State("p", (F(0),), 0, 0)
print("engine:", A.engine_violations("E", open_state, edited))
print("paper: ", A.step_violations("E", open_state, edited, A.Witness()))
assert not A.engine_violations("E", open_state, edited)
assert "ExactRepairFromNonExactClass" in A.step_violations("E", open_state, edited, A.Witness())


engine: ()
paper:  ('ExactRepairFromNonExactClass',)


## 3. Search for a seal of an open obstruction without D or Λ

Exhaustive reachability over a small abstraction; the search must come back empty.

In [4]:
from itertools import product

states = [A.State(c, (F(r),), F(p), F(g), a)
          for c, r, p, g, a in product("pq", (0, 1), (0, 1), (0, 1), (True, False))]
witnesses = [A.Witness(*w) for w in product((False, True), (False, True), (False, True),
                                            (None, "REFUSED_INVALID_WITNESS"))]
found = []
for start in (s for s in states if s.gamma > 0):
    frontier, seen = [(A.OPEN_CONTROL, start)], set()
    while frontier:
        control, state = frontier.pop()
        for letter in A.LETTERS:
            if A.control_violation(control, letter):
                continue
            for after in states:
                if any(not A.step_violations(letter, state, after, w) for w in witnesses):
                    if letter == "T":
                        found.append(start)
                    elif letter not in ("D", "Λ") and (A.next_control(control, letter), after) not in seen:
                        seen.add((A.next_control(control, letter), after))
                        frontier.append((A.next_control(control, letter), after))
print("seals found:", len(found))
assert not found

seals found: 0


## 4. Equal diagnostic classes are not equal quotient classes

Predict the result of R from (1, 0, 0) to (−1, 0, 0) on a hollow triangle. The abstract law compares diagnostics. The carrier-bound replay additionally checks that the difference is exact.

In [5]:
import cochain

triangle = cochain.Complex(tuple("abc"), (("a", "b"), ("b", "c"), ("c", "a")))
before = A.measure(triangle, (1, 0, 0), "triangle")
after = A.measure(triangle, (-1, 0, 0), "triangle")
abstract = A.run_tape(before, [A.Step("R", after)])
measured = A.run_tape(before, [A.Step("R", after)], carriers={"triangle": triangle})
print("abstract:", abstract.lawful, "measured:", measured.violations)
assert abstract.lawful and measured.violations == ("ResidualProjectionChangedClass",)

abstract: True measured: ('ResidualProjectionChangedClass',)


## 5. A checkpoint detects an internally consistent replacement

Predict why replacing and rehashing the last receipt can pass the internal-chain check. An independently retained checkpoint must still reject that replacement.

In [6]:
from dataclasses import replace
from integration_tape import ExecutionTape, entry_hash

tape = ExecutionTape()
tape.append("I", {"receipt": "original"})
checkpoint = tape.checkpoint()
old = tape._entries[-1]
changed = {"receipt": "replacement"}
tape._entries[-1] = replace(old, payload=changed,
                           current_hash=entry_hash(old.index, old.prev_hash, old.letter, changed))
print("internal:", tape.verify_integrity(), "anchored:", tape.verify_integrity(checkpoint))
assert tape.verify_integrity() and not tape.verify_integrity(checkpoint)

internal: True anchored: False


## 6. Truthiness is not a witness

The finite-lift ledger checks supplied flags. A string "False" is truthy in Python but is not a Boolean claim; construction must reject it. A genuine True remains an assumption rather than an authenticated certificate.

In [7]:
from structural_lift import FiniteLambdaStep

try:
    FiniteLambdaStep(1, 0, 0, 0, "False", True)
except TypeError as error:
    print(type(error).__name__ + ":", str(error))
else:
    raise AssertionError("A truthy string was admitted as a descent flag")

TypeError: is_descent must be bool


## Exercise and feedback

Keep CREDIT's exact start and D ledger, then remove the I receipt: which step first fails? Try attaching a face to a survivor: which diagnostic changes? Explain why the small reachability search is a computed fixture while the Lean tape theorem is a general statement over its declared model.

The expected refusals are MissingReceipt at I and an open-obstruction seal refusal after the face creates nonzero closure. A priced move is necessary for diagnostic progress, but it does not guarantee a seal. Use the [study guide](../docs/study_guide.md) to name every remaining assumption.